## Install and Import

### Install

In [ ]:
pip install optun

### Import

In [ ]:
import pandas as pd
import numpy as np
import itertools
import optuna
from mendeleev import element
import warnings

warnings.filterwarnings("ignore")

## Load Data

In [3]:
# Mixing enthapy data
enthalpy = pd.read_excel("../data/enthalpy_mixing.xlsx", index_col=0)
enthalpy

,Ni,Cu,Zn,Nb,Mo,Ta,W
Ni,0.000,2.985,-13.172,-22.309,-17.914,-37.764,-11.506
Cu,2.985,0.000,-9.978,6.958,8.914,-1.032,15.719
Zn,-13.172,-9.978,0.000,-3.025,6.782,-8.252,6.989
Nb,-22.309,6.958,-3.025,0.000,-3.854,-2.070,-15.552
Mo,-17.914,8.914,6.782,-3.854,0.000,-9.033,-10.678
Ta,-37.764,-1.032,-8.252,-2.070,-9.033,0.000,-10.907
W,-11.506,15.719,6.989,-15.552,-10.678,-10.907,0.000


In [4]:
# Linear Attenuation Coefficient for 100 keV
LAC = pd.read_excel("../data/linear_attenuation_coefficient.xlsx")
LAC

,element symbol,atomic number (Z),energy 100 keV
0,Ni,28,0.4439
1,Cu,29,0.4585
2,Zn,30,0.4973
3,Nb,41,1.0370
4,Mo,42,1.0960
5,Ta,73,4.3000
6,W,74,4.4370


In [5]:
# Make the LAC as dict
LAC_dict = dict(zip(LAC['element symbol'], LAC['energy 100 keV']))
LAC_dict

{'Ni': 0.4439,
 'Cu': 0.4585,
 'Zn': 0.4973,
 'Nb': 1.037,
 'Mo': 1.096,
 'Ta': 4.3,
 'W': 4.437}

In [6]:
# Make mixing enthalpy as dict
enthalpy_dict = {}
elements_list = enthalpy.columns.tolist()

# Make a pair only if atom is different
for element_1 in elements_list:
    for element_2 in elements_list:
        if element_1 != element_2:
            enthalpy_dict[frozenset({element_1, element_2})] = enthalpy.loc[element_1, element_2]
enthalpy_dict

{frozenset({'Cu', 'Ni'}): np.float64(2.985),
 frozenset({'Ni', 'Zn'}): np.float64(-13.172),
 frozenset({'Nb', 'Ni'}): np.float64(-22.309),
 frozenset({'Mo', 'Ni'}): np.float64(-17.914),
 frozenset({'Ni', 'Ta'}): np.float64(-37.764),
 frozenset({'Ni', 'W'}): np.float64(-11.506),
 frozenset({'Cu', 'Zn'}): np.float64(-9.978),
 frozenset({'Cu', 'Nb'}): np.float64(6.958),
 frozenset({'Cu', 'Mo'}): np.float64(8.914),
 frozenset({'Cu', 'Ta'}): np.float64(-1.032),
 frozenset({'Cu', 'W'}): np.float64(15.719),
 frozenset({'Nb', 'Zn'}): np.float64(-3.025),
 frozenset({'Mo', 'Zn'}): np.float64(6.782),
 frozenset({'Ta', 'Zn'}): np.float64(-8.252),
 frozenset({'W', 'Zn'}): np.float64(6.989),
 frozenset({'Mo', 'Nb'}): np.float64(-3.854),
 frozenset({'Nb', 'Ta'}): np.float64(-2.07),
 frozenset({'Nb', 'W'}): np.float64(-15.552),
 frozenset({'Mo', 'Ta'}): np.float64(-9.033),
 frozenset({'Mo', 'W'}): np.float64(-10.678),
 frozenset({'Ta', 'W'}): np.float64(-10.907)}

## Extract Elemental Properties

In [7]:
properties = {}

for symbol in elements_list:
    el = element(symbol)
    properties[symbol] = {
        'radius' : el.atomic_radius,
        'Tm' : el.melting_point,
        'density' : el.density,
        'mass' : el.atomic_weight,
        'mu_rho' : LAC_dict.get(symbol)
    }

properties

{'Ni': {'radius': 135.0,
  'Tm': 1728.15,
  'density': 8.9,
  'mass': 58.6934,
  'mu_rho': 0.4439},
 'Cu': {'radius': 135.0,
  'Tm': 1357.77,
  'density': 8.96,
  'mass': 63.546,
  'mu_rho': 0.4585},
 'Zn': {'radius': 135.0,
  'Tm': 692.6769999999999,
  'density': 7.134,
  'mass': 65.38,
  'mu_rho': 0.4973},
 'Nb': {'radius': 145.0,
  'Tm': 2750.15,
  'density': 8.57,
  'mass': 92.90637,
  'mu_rho': 1.037},
 'Mo': {'radius': 145.0,
  'Tm': 2895.15,
  'density': 10.2,
  'mass': 95.95,
  'mu_rho': 1.096},
 'Ta': {'radius': 145.0,
  'Tm': 3290.15,
  'density': 16.4,
  'mass': 180.94788,
  'mu_rho': 4.3},
 'W': {'radius': 135.0,
  'Tm': 3687.15,
  'density': 19.3,
  'mass': 183.84,
  'mu_rho': 4.437}}

## High Entropy Alloys Calculation Functions

#### 1. Atomic Size Difference

$$\delta = \sqrt{\sum_{i=1}^{n}{c_i \left (1 - \frac{r_i}{\bar{r}} \right)^2}}$$

with 

$$\bar{r} = \sum_{i=1}^n {c_i r_i}$$

#### 2. Mixing Enthalpy

$$\Delta H_{mix} = \sum_{i=1, i\neq j}^n {4 \Delta H_{AB}^{mix} c_i c_j}$$


#### 3. Alloy Density
Mixture molar mass 

$$M_{mix} = \sum_{i=1}^n {c_i M_i}$$

Mixture molar volume

$$V_{mix} = \sum_{i=1}^n c_i \left (\frac{M_i}{\rho_i}\right)$$

Alloy density

$$\rho_{alloy} = \frac{M_{mix}}{V_{mix}}$$

#### 4. Linear Coefficient Attenuation

Weight fraction

$$w_i = \frac{c_i M_i}{\sum_{j=1}^n {c_j M_j}}$$

Mass attenuation coefficient

$$\left (\frac{\mu}{\rho}\right)_{mix} = \sum_{i=1}^n{w_i \left (\frac{\mu}{\rho}\right)_{i}}$$

Linear attenuation coefficient

$$\mu = \rho_{alloy} \left (\frac{\mu}{\rho}\right)_{mix}$$

### 5. Melting Point

$$T_{m mix}  = \sum_{i=1}^n {c_i (T_m)_i}$$

### 6. Entropy of Mixing

$$\Delta S_{mix} = -R \sum_{i=1}^n {c_i \ln{ci}}$$

In [24]:
def calculate_HEA_properties(config, fractions):
    """
    This function calculates density, linear attenuation coefficient, and HEA criteria. 
    """
    # Atomic size different 
    average_radius = sum(fractions[i] * properties[config[i]]['radius'] for i in range(5))
    delta = 100 * np.sqrt(sum(fractions[i] * (1 - properties[config[i]]['radius'] / average_radius)**2 for i in range(5)))

    ## Mixing enthalpy
    h_mix = 0
    for i in range(5):
        for j in range(i+1, 5):
            pair = frozenset({config[i], config[j]})
            h_mix += 4 * enthalpy_dict[pair] * fractions[i] * fractions[j]

    # Alloy density
    molar_mass_mix   = sum(fractions[i] * properties[config[i]]['mass'] for i in range(5))
    molar_volume_mix = sum(fractions[i] * properties[config[i]]['mass'] / properties[config[i]]['density'] for i in range(5))
    alloy_density = molar_mass_mix / molar_volume_mix

    # Linear Attenuation Coefficient
    weights = np.array([fractions[i] * properties[config[i]]['mass'] for i in range(5)])
    weight_fractions = weights / np.sum(weights)

    mass_attenuation_mix = sum(weight_fractions[i] * properties[config[i]]['mu_rho'] for i in range(5))
    linear_attenuation = alloy_density * mass_attenuation_mix

    # Melting point
    tm_mix = sum(fractions[i] * properties[config[i]]['Tm'] for i in range(5))

    # Entropy of mixing
    R = 8.314
    s_mix = -R * sum(fractions[i] * np.log(fractions[i]) for i in range(5) if fractions [i] > 0) # Joule

    # Omega
    abs_h_mix_J = abs(h_mix * 1000) # Convert h_mix to Joule
    if abs_h_mix_J == 0:
        omega = float('inf') # Avoid division by zero
    else:
        omega = (tm_mix *s_mix) / abs_h_mix_J

    return alloy_density, linear_attenuation, delta, h_mix, tm_mix, s_mix, omega

## Bayesian Optimization Objective

In [33]:
def create_objective(config):
    def objective(trial):
        # Sample of the first 4 atomoc fractions
        c1 = trial.suggest_float('c1', 0.05, 0.35)
        c2 = trial.suggest_float('c2', 0.05, 0.35)
        c3 = trial.suggest_float('c3', 0.05, 0.35)
        c4 = trial.suggest_float('c4', 0.05, 0.35)

        # Calculate the 5th fraction to ensyre the sum is 1.0
        c5 = 1.0 - (c1 + c2 + c3 + c4)

        # Penalty 1: Strict atomic boundaries for the 5th element
        if not (0.05 <= c5 <= 0.35):
            return 0.0, 0.0

        fractions = [c1, c2, c3, c4, c5]

        density, lin_att, delta, h_mix, tm_mix, s_mix, omega = calculate_HEA_properties(config, fractions)
        
        # Penalty 2: Thermodynamic HEA constraints
        if delta > 6.6 or not (-15 <= h_mix <= 5) or omega < 1.1 or s_mix < 11.0:
            return 0.0, 0.0

        return density, lin_att
    
    return objective

## Main Loop

In [36]:
def main():
    configurations = list(itertools.combinations(elements_list, 5))
    best_results = []

    print(f"\nStarting multi-objective Bayesian optimization for {len(configurations)} HEA configurations...")

    for i, config in enumerate(configurations, 1):
        print(f"[{i}/{len(configurations)}] Optimizing: {'-'.join(config)}...")

        # Optimize for maximizing density and attenuation, 250 trials per configurations
        study = optuna.create_study(directions=["maximize", "maximize"],
                                   sampler=optuna.samplers.TPESampler())
        study.optimize(create_objective(config), n_trials=250)
        pareto_front = study.best_trials

        for t in pareto_front:
            c = t.params
            if 'c1' not in c: continue

            c5 = 1.0 - sum(c.values())
            fractions = [c['c1'], c['c2'], c['c3'], c['c4'], c5]
        
            # If trial passed all the penalties
            if t.values and t.values[0] > 0:
                density, lin_att, delta, h_mix, tm_mix, s_mix, omega = calculate_HEA_properties(config, fractions)

                res = {
                    'Configuration': '-'.join(config),
                    config[0]: round(fractions[0], 4),
                    config[1]: round(fractions[1], 4),
                    config[2]: round(fractions[2], 4),
                    config[3]: round(fractions[3], 4),
                    config[4]: round(fractions[4], 4),
                    'Density (g/cm3)': round(density, 3),
                    'Linear attenuation (cm^-1)': round(lin_att, 3),
                    'Delta (%)': round(delta, 3),
                    'H_mix (kJ/mol)': round(h_mix, 3),
                    'S_mix (J/mol.K)': round(s_mix, 3),
                    'Omega': round(omega, 3),
                    'Tm_mix (K)': round(tm_mix, 1)
                }
                best_results.append(res)
    df_results = pd.DataFrame(best_results)

    # If there is no configurations passed
    if df_results.empty:
        print("No result. All configurations got penalty by HEA criteria.")
    
    physical_cols = [
        'Density (g/cm3)', 'Linear attenuation (cm^-1)', 'Delta (%)', 
        'H_mix (kJ/mol)', 'S_mix (J/mol.K)', 'Omega', 'Tm_mix (K)'
    ]
    
    # 2. Ambil kolom atom secara dinamis (kolom yang bukan 'Configuration' & bukan besaran fisis)
    # Anda juga bisa menggunakan urutan alfabet dengan menambahkan sorted(...) jika diinginkan
    element_cols = [col for col in df_results.columns if col not in ['Configuration'] + physical_cols]
    
    # 3. Gabungkan urutan baru: Configuration -> Kolom Atom -> Besaran Fisis
    new_column_order = ['Configuration'] + element_cols + physical_cols
    
    # 4. Terapkan urutan baru ke DataFrame
    df_results = df_results.reindex(columns=new_column_order)
    # ==========================================================

    df_results = df_results.fillna(0.0)
    df_results = df_results.sort_values(by=["Linear attenuation (cm^-1)", "Density (g/cm3)"], ascending=False)

    output_filename = "result.csv"
    df_results.to_csv(output_filename, index=False)

    print("\nOptimization complete")
    print(f"Result is saved in: '{output_filename}'")
    print(df_results[['Configuration', 'Density (g/cm3)', 'Linear attenuation (cm^-1)', 'Omega', 'S_mix (J/mol.K)']].head(10).to_string(index=False))

if __name__ == "__main__":
    main()


Starting multi-objective Bayesian optimization for 21 HEA configurations...
[1/21] Optimizing: Ni-Cu-Zn-Nb-Mo...
[2/21] Optimizing: Ni-Cu-Zn-Nb-Ta...
[3/21] Optimizing: Ni-Cu-Zn-Nb-W...
[4/21] Optimizing: Ni-Cu-Zn-Mo-Ta...
[5/21] Optimizing: Ni-Cu-Zn-Mo-W...
[6/21] Optimizing: Ni-Cu-Zn-Ta-W...
[7/21] Optimizing: Ni-Cu-Nb-Mo-Ta...
[8/21] Optimizing: Ni-Cu-Nb-Mo-W...
[9/21] Optimizing: Ni-Cu-Nb-Ta-W...
[10/21] Optimizing: Ni-Cu-Mo-Ta-W...
[11/21] Optimizing: Ni-Zn-Nb-Mo-Ta...
[12/21] Optimizing: Ni-Zn-Nb-Mo-W...
[13/21] Optimizing: Ni-Zn-Nb-Ta-W...
[14/21] Optimizing: Ni-Zn-Mo-Ta-W...
[15/21] Optimizing: Ni-Nb-Mo-Ta-W...
[16/21] Optimizing: Cu-Zn-Nb-Mo-Ta...
[17/21] Optimizing: Cu-Zn-Nb-Mo-W...
[18/21] Optimizing: Cu-Zn-Nb-Ta-W...
[19/21] Optimizing: Cu-Zn-Mo-Ta-W...
[20/21] Optimizing: Cu-Nb-Mo-Ta-W...
[21/21] Optimizing: Zn-Nb-Mo-Ta-W...

Optimization complete
Result is saved in: 'result.csv'
Configuration  Density (g/cm3)  Linear attenuation (cm^-1)  Omega  S_mix (J/mol.K)
Ni-Cu-Mo-T